# Sampel → Fitur → Model → Peta
Notebook ini memakai `s2_aoi_20m.tif` (hasil openEO) dan `training_samples_clean.geojson` (sampel digitasi yang sudah dibersihkan).

Urutan: ekstraksi piksel → indeks → analisis band per kelas → model (Random Forest vs XGBoost, uji **per poligon**) → klasifikasi seluruh raster → poligon GeoJSON per kelas.

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from rasterio.windows import Window
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

RASTER   = 's2_aoi_20m.tif'
SAMPLES  = 'training_samples_clean.geojson'
OUT_DIR  = 'outputs'
SCALE    = 10000.0          # raster openEO = bilangan bulat (reflektansi x 10000)
MAX_PX_PER_POLY = 300       # batasi piksel per poligon agar poligon besar tidak mendominasi
RANDOM_STATE = 42
RGB_ORDER = ('B8A', 'B11', 'B4')    # urutan R, G, B komposit; ganti kalau dosen berbeda
BAND_NAMES = ['B2', 'B3', 'B4', 'B8', 'B8A', 'B11', 'B12']
os.makedirs(OUT_DIR, exist_ok=True)

# id kelas harus sama dengan alat digitasi: kode -> (label, warna)
CLASSES = {
    1: ('built_up', 'Bangunan',        '#e53935'),
    2: ('farmland', 'Lahan pertanian', '#fdd835'),
    3: ('water',    'Badan air',       '#1e88e5'),
    4: ('forest',   'Hutan',           '#2e7d32'),
    5: ('mangrove', 'Hutan mangrove',  '#00e5c0'),
}
CODE2ID = {v[0]: k for k, v in CLASSES.items()}
NAMES = [CLASSES[i][1] for i in range(1, 6)]

In [ ]:
with rasterio.open(RASTER) as src:
    print(src.count, 'band,', src.width, 'x', src.height, src.crs, src.res, src.dtypes[0], 'nodata =', src.nodata)
    NODATA, R_CRS, R_RES = src.nodata, src.crs, src.res
    assert src.count == 7, 'raster harus 7 band: B2,B3,B4,B8,B8A,B11,B12'

samples = gpd.read_file(SAMPLES).to_crs(R_CRS).reset_index(drop=True)
samples['class_id'] = samples['class'].map(CODE2ID)
assert samples['class_id'].notna().all(), f"kelas tak dikenal: {samples.loc[samples['class_id'].isna(), 'class'].unique()}"
samples['poly_id'] = samples.index
print(samples['class'].value_counts())

## 1. Ekstraksi piksel + indeks

In [ ]:
EPS = 1e-6
FEATURES = BAND_NAMES + ['NDVI', 'NDWI', 'NDBI', 'MNDWI', 'NDMI']

def nd(a, b):
    return (a - b) / (a + b + EPS)

def feature_stack(refl):
    """refl: (7, ...) reflektansi 0-1  ->  (12, ...) = 7 band + 5 indeks"""
    B2, B3, B4, B8, B8A, B11, B12 = refl
    idx = np.stack([nd(B8, B4),     # NDVI  vegetasi
                    nd(B3, B8),     # NDWI  air
                    nd(B11, B8),    # NDBI  lahan terbangun
                    nd(B3, B11),    # MNDWI air (SWIR)
                    nd(B8A, B11)])  # NDMI  kelembapan vegetasi
    return np.concatenate([refl, idx], axis=0)

rng = np.random.default_rng(RANDOM_STATE)
parts = []
with rasterio.open(RASTER) as src:
    for _, r in samples.iterrows():
        for all_touched in (False, True):          # poligon sempit: coba all_touched
            data, tr = mask(src, [r.geometry], crop=True, all_touched=all_touched, filled=False)
            inside = ~np.ma.getmaskarray(data).any(axis=0)
            valid = inside if NODATA is None else inside & (data.data != NODATA).all(axis=0)
            if valid.sum() > 0:
                break
        ys, xs = np.nonzero(valid)
        if len(ys) == 0:
            print('poligon tanpa piksel valid:', r['poly_id'], r['class']); continue
        if len(ys) > MAX_PX_PER_POLY:
            sel = rng.choice(len(ys), MAX_PX_PER_POLY, replace=False); ys, xs = ys[sel], xs[sel]
        refl = np.clip(data.data[:, ys, xs].astype(np.float32) / SCALE, 0, 1)
        feats = feature_stack(refl)
        gx, gy = rasterio.transform.xy(tr, ys, xs)
        d = pd.DataFrame(feats.T, columns=FEATURES)
        d.insert(0, 'y', gy); d.insert(0, 'x', gx)
        d.insert(0, 'class_id', int(r['class_id'])); d.insert(0, 'class', r['class']); d.insert(0, 'poly_id', int(r['poly_id']))
        parts.append(d)

df = pd.concat(parts, ignore_index=True)
df.to_csv(f'{OUT_DIR}/training_pixels.csv', index=False)
print(len(df), 'piksel ->', f'{OUT_DIR}/training_pixels.csv')
print(df.groupby('class').agg(piksel=('poly_id', 'size'), poligon=('poly_id', 'nunique')))

## 2. Tanda spektral dan band paling membedakan

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for cid, (code_, label, color) in CLASSES.items():
    sub = df[df.class_id == cid][BAND_NAMES]
    m, s = sub.mean(), sub.std()
    ax.plot(BAND_NAMES, m, marker='o', color=color, label=label)
    ax.fill_between(BAND_NAMES, m - s, m + s, color=color, alpha=0.15)
ax.set_ylabel('Reflektansi'); ax.set_title('Tanda spektral rata-rata ± simpangan baku'); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/tanda_spektral.png', dpi=150); plt.show()

In [ ]:
from sklearn.metrics import roc_auc_score

# Seberapa baik tiap fitur memisahkan satu kelas dari sisanya (0.5 = tidak berguna, 1.0 = sempurna)
sep = pd.DataFrame({
    CLASSES[cid][1]: {f: max(roc_auc_score(df.class_id == cid, df[f]), 1 - roc_auc_score(df.class_id == cid, df[f])) for f in FEATURES}
    for cid in CLASSES
}).round(3)
display(sep)

# Khusus mangrove vs hutan
sub = df[df.class_id.isin([4, 5])]
mh = {f: max(roc_auc_score(sub.class_id == 5, sub[f]), 1 - roc_auc_score(sub.class_id == 5, sub[f])) for f in FEATURES}
print('Mangrove vs Hutan, fitur terbaik:')
print(pd.Series(mh).sort_values(ascending=False).round(3).head(8))
sep.to_csv(f'{OUT_DIR}/separabilitas_fitur.csv')

In [ ]:
# Bandingkan komposit untuk satu contoh poligon per kelas
def window_around(src, geom, half=40):
    row, col = src.index(geom.centroid.x, geom.centroid.y)
    arr = src.read(window=Window(col - half, row - half, 2 * half, 2 * half), boundless=True,
                   fill_value=NODATA if NODATA is not None else 0).astype(np.float32)
    if NODATA is not None:
        arr[arr == NODATA] = np.nan
    return arr / SCALE

def comp(arr, bands):
    img = np.dstack([arr[BAND_NAMES.index(b)] for b in bands])
    lo, hi = np.nanpercentile(img, (2, 98))
    return np.clip((img - lo) / (hi - lo + 1e-9), 0, 1)

COMPS = {'Warna alami (B4-B3-B2)': ('B4', 'B3', 'B2'),
         f'Komposit tugas ({"-".join(RGB_ORDER)})': RGB_ORDER,
         'Vegetasi (B8-B4-B3)': ('B8', 'B4', 'B3')}
fig, axes = plt.subplots(5, 3, figsize=(10, 16))
with rasterio.open(RASTER) as src:
    for i, cid in enumerate(CLASSES):
        g = samples[samples.class_id == cid].sample(1, random_state=RANDOM_STATE).geometry.iloc[0]
        arr = window_around(src, g)
        for j, (title, bands) in enumerate(COMPS.items()):
            axes[i, j].imshow(comp(arr, bands)); axes[i, j].axis('off')
            axes[i, j].set_title(f'{CLASSES[cid][1]}\n{title}', fontsize=8)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/perbandingan_komposit.png', dpi=130); plt.show()

## 3. Model dan evaluasi
Data dibagi **per poligon** (bukan per piksel), supaya piksel bertetangga dari poligon yang sama tidak muncul di data latih *dan* uji sekaligus. Pembagian tetap menjaga proporsi kelas.

In [ ]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from xgboost import XGBClassifier

X = df[FEATURES].values.astype(np.float32)
y = df['class_id'].values - 1          # 0..4
g = df['poly_id'].values

# hold-out uji 20% poligon
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
tr, te = next(sgkf.split(X, y, g))
X_tr, X_te, y_tr, y_te, g_tr = X[tr], X[te], y[tr], y[te], g[tr]
print('piksel latih/uji :', len(tr), len(te))
print('poligon uji per kelas:\n', df.iloc[te].groupby('class').poly_id.nunique().to_string())

def make_model(name):
    if name == 'RandomForest':
        return RandomForestClassifier(n_estimators=300, class_weight='balanced_subsample', n_jobs=-1, random_state=RANDOM_STATE)
    return XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
                         tree_method='hist', n_jobs=-1, random_state=RANDOM_STATE)

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rows = []
for name in ['RandomForest', 'XGBoost']:
    res = cross_validate(make_model(name), X_tr, y_tr, groups=g_tr, cv=cv, scoring=['accuracy', 'f1_macro'])
    rows.append({'model': name, 'akurasi_cv': res['test_accuracy'].mean(), 'std': res['test_accuracy'].std(), 'f1_macro_cv': res['test_f1_macro'].mean()})
cv_table = pd.DataFrame(rows).round(4)
display(cv_table)
BEST = cv_table.sort_values('f1_macro_cv', ascending=False).iloc[0]['model']
print('Model terpilih:', BEST)

In [ ]:
import joblib

model = make_model(BEST).fit(X_tr, y_tr)
pred = model.predict(X_te)
acc = accuracy_score(y_te, pred)
f1m = f1_score(y_te, pred, average='macro')
print(f'Akurasi uji = {acc:.4f} | F1 macro = {f1m:.4f}\n')
print(classification_report(y_te, pred, labels=range(5), target_names=NAMES, digits=3, zero_division=0))

cm = confusion_matrix(y_te, pred, labels=range(5))
fig, ax = plt.subplots(figsize=(6.5, 5.5))
ConfusionMatrixDisplay(cm, display_labels=NAMES).plot(ax=ax, cmap='Blues', xticks_rotation=30, colorbar=False)
ax.set_title(f'Confusion matrix ({BEST}, uji per poligon)')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/confusion_matrix.png', dpi=150); plt.show()

imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values()
imp.plot.barh(figsize=(6, 4), title='Kepentingan fitur'); plt.tight_layout()
plt.savefig(f'{OUT_DIR}/feature_importance.png', dpi=150); plt.show()

metrics = {
    'model': BEST, 'akurasi_uji': float(acc), 'f1_macro_uji': float(f1m),
    'cv': cv_table.to_dict('records'),
    'laporan': classification_report(y_te, pred, labels=range(5), target_names=NAMES, output_dict=True, zero_division=0),
    'piksel_latih': int(len(tr)), 'piksel_uji': int(len(te)),
    'poligon_per_kelas': df.groupby('class').poly_id.nunique().to_dict(),
}
json.dump(metrics, open(f'{OUT_DIR}/metrics.json', 'w'), indent=2, ensure_ascii=False)

# model akhir untuk peta: dilatih ulang dengan SEMUA sampel (angka akurasi di atas tetap dari model hold-out)
final = make_model(BEST).fit(X, y)
joblib.dump(final, f'{OUT_DIR}/model.joblib')

## 4. Klasifikasi seluruh raster

In [ ]:
CLS_TIF = f'{OUT_DIR}/klasifikasi_20m.tif'
ROWS = 256
with rasterio.open(RASTER) as src:
    profile = dict(driver='GTiff', height=src.height, width=src.width, count=1, dtype='uint8',
                   crs=src.crs, transform=src.transform, nodata=0, compress='deflate')
    counts = np.zeros(6, dtype=np.int64)
    with rasterio.open(CLS_TIF, 'w', **profile) as dst:
        for r0 in range(0, src.height, ROWS):
            h = min(ROWS, src.height - r0)
            win = Window(0, r0, src.width, h)
            arr = src.read(window=win)
            valid = np.ones(arr.shape[1:], bool) if NODATA is None else (arr != NODATA).all(axis=0)
            out = np.zeros(arr.shape[1:], np.uint8)
            if valid.any():
                refl = np.clip(arr[:, valid].astype(np.float32) / SCALE, 0, 1)
                out[valid] = final.predict(feature_stack(refl).T).astype(np.uint8) + 1
            counts += np.bincount(out.ravel(), minlength=6)
            dst.write(out, 1, window=win)
            if (r0 // ROWS) % 10 == 0:
                print(f'baris {r0}/{src.height}', end='\r')

px_km2 = R_RES[0] * R_RES[1] / 1e6
area = pd.DataFrame({'kelas': NAMES, 'kode': [CLASSES[i][0] for i in range(1, 6)],
                     'piksel': counts[1:], 'luas_km2': (counts[1:] * px_km2).round(2)})
area['persen'] = (100 * area.piksel / area.piksel.sum()).round(2)
area.to_csv(f'{OUT_DIR}/luas_per_kelas.csv', index=False)
display(area)

## 5. Poligon GeoJSON per kelas (untuk Leaflet)
Hasil raster dibersihkan dari bercak kecil (`SIEVE_PX`), diubah jadi poligon, disederhanakan, dan disimpan per kelas dalam EPSG:4326.
Naikkan `SIEVE_PX` / `SIMPLIFY_M` kalau file terlalu besar.

In [ ]:
from rasterio.features import sieve, shapes
from shapely.geometry import shape

SIEVE_PX = 50        # buang bercak < 50 piksel (= 2 ha)
SIMPLIFY_M = 20      # penyederhanaan garis (meter)

with rasterio.open(CLS_TIF) as src:
    cls = src.read(1); tfm = src.transform; crs = src.crs
cls = sieve(cls, size=SIEVE_PX, mask=cls > 0, connectivity=4)

recs = [(shape(geom), int(v)) for geom, v in shapes(cls, mask=cls > 0, transform=tfm)]
poly = gpd.GeoDataFrame({'class_id': [v for _, v in recs]}, geometry=[g_ for g_, _ in recs], crs=crs)
poly['class'] = poly.class_id.map(lambda i: CLASSES[i][0])
poly['label'] = poly.class_id.map(lambda i: CLASSES[i][1])
poly['luas_ha'] = (poly.geometry.area / 1e4).round(2)
poly['geometry'] = poly.geometry.simplify(SIMPLIFY_M, preserve_topology=True)
poly = poly.to_crs(4326)
try:
    poly['geometry'] = poly.geometry.set_precision(1e-5)     # kurangi digit koordinat -> file lebih kecil
except Exception:
    pass

for cid, (code_, label, _) in CLASSES.items():
    part = poly[poly.class_id == cid]
    path = f'{OUT_DIR}/hasil_{code_}.geojson'
    part.to_file(path, driver='GeoJSON')
    print(f'{label:18s} {len(part):7d} poligon  {os.path.getsize(path) / 1e6:7.1f} MB')